# Programming for Data Analytics Big Project: Irish Wind Resource Analysis
This notebook uses hourly Met Éireann weather data to assess wind energy potential in Ireland.

Data: [Met Éireann Historical Data](https://www.met.ie/climate/available-data/historical-data)

In [1]:
import pandas as pd
from pathlib import Path

## Loading the data
The file `malin_head_hourly.csv` has 40 lines of station metadata above the csv header, so these lines are skipped when loading.

- **Mean Wind speed** (`wdsp`) is the mean hourly speed in knots. its is converted to m/s (1 knot = 0.514444 m/s).
- **`ind` columns** are quality codes. Each one describes the variable that follows it: rainfall, dry bulb temperature, wet bulb temperature, wind speed and wind direction.

In [2]:
df = pd.read_csv("data/malin_head_hourly.csv", skiprows=41, skipinitialspace=True, low_memory=False,)
df.columns.tolist()

['date',
 'ind',
 'rain',
 'ind.1',
 'temp',
 'ind.2',
 'wetb',
 'dewpt',
 'vappr',
 'rhum',
 'msl',
 'ind.3',
 'wdsp',
 'ind.4',
 'wddir',
 'ww',
 'w',
 'sun',
 'vis',
 'clht',
 'clamt']

In [3]:
df = df.rename(columns={ "ind": "rain_ind", "ind.1": "temp_ind", "ind.2": "wetb_ind", "ind.3": "wdsp_ind", "ind.4": "wddir_ind",})
df["date"] = pd.to_datetime(df["date"], format="%d-%b-%Y %H:%M")
df["wind_ms"] = df["wdsp"] * 0.514444
df.head()

,date,rain_ind,rain,temp_ind,temp,wetb_ind,wetb,dewpt,vappr,rhum,...,wdsp,wddir_ind,wddir,ww,w,sun,vis,clht,clamt,wind_ms
0,1955-05-01 01:00:00,0,0.0,0,8.5,0,7.2,5.5,9.1,83.0,...,11.0,0,130.0,3.0,2.0,0.0,30000.0,40.0,7.0,5.658884
1,1955-05-01 02:00:00,0,0.0,0,8.2,0,7.2,6.1,9.3,85.0,...,11.0,0,120.0,1.0,2.0,0.0,30000.0,100.0,6.0,5.658884
2,1955-05-01 03:00:00,3,0.0,0,7.3,0,6.6,5.5,9.3,91.0,...,8.0,0,120.0,1.0,2.0,0.0,30000.0,220.0,5.0,4.115552
3,1955-05-01 04:00:00,3,0.0,0,7.8,0,7.2,6.1,9.6,91.0,...,18.0,0,130.0,3.0,2.0,0.0,30000.0,90.0,7.0,9.259992
4,1955-05-01 05:00:00,0,0.0,0,8.1,0,7.3,6.6,9.7,89.0,...,9.0,0,110.0,2.0,2.0,0.0,40000.0,90.0,7.0,4.629996


In [4]:
print(df.shape)
print(df["date"].min(), df["date"].max())

(626064, 22)
1955-05-01 01:00:00 2026-10-01 00:00:00


The Malin Head file has 626,064 hourly observation from 1 May 1955 to 1 October 2026.

## Loading all stations
The station files have different header lengths and column counts,so a function finds the header line in each file and renames the duplicate `ind` columns after the variable that follows them. Only the columns used in the anlysis are kept, and a `station` column indentifies each file.

In [5]:
DATA_DIR = Path("data")
KEEP = ["date", "wdsp", "wdsp_ind", "wddir", "temp", "rain", "msl"]

def load_station(path):
    with open(path) as f:
        for i, line in enumerate(f):
            if line.startswith("date,"):
                header_line, columns = i, line.strip().split(",")
                break
            
    columns = [f"{columns[j + 1]}_ind" if c == "ind" else c for j, c in enumerate(columns)]
    df = pd.read_csv(path, skiprows=header_line + 1, header=None,  names=columns, skipinitialspace=True, low_memory=False)
    
    df["date"] = pd.to_datetime(df["date"], format="%d-%b-%Y %H:%M")
    df["station"] = path.stem.removesuffix("_hourly")
    df = df[["station"] + KEEP]
    df[KEEP[1:]] = df[KEEP[1:]].apply(pd.to_numeric, errors="coerce")
    return df

In [6]:
frames = [load_station(p) for p in sorted(DATA_DIR.glob("*_hourly.csv"))]
df = pd.concat(frames, ignore_index=True)
df["winds_ms"] = df["wdsp"] * 0.514444

df.groupby("station")["date"].agg(["count", "min", "max"])

,count,min,max
station,,,
belmullet,613954,1956-09-16 15:00:00,2026-10-01
cork_airport,567576,1962-01-01 01:00:00,2026-10-01
dublin_airport,707833,1946-01-01 00:00:00,2026-10-01
gurteen,163704,2007-12-31 01:00:00,2026-10-01
malin_head,626064,1955-05-01 01:00:00,2026-10-01
mullingar,463704,1973-11-07 01:00:00,2026-10-01
shannon_airport,707833,1946-01-01 00:00:00,2026-10-01
valentia_observatory,707832,1946-01-01 00:00:00,2026-10-01
